#### source: https://github.com/Lyken17/pytorch-OpCounter

In [7]:
import torch
from thop import profile, clever_format

In [2]:
from models.onedcnn_model_16frames import CNN_ForecastNet

model = CNN_ForecastNet()
input = torch.randn(1,16,5)
macs, params = profile(model, inputs=(input, ))

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv1d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.


In [3]:
total_params = sum(
	param.numel() for param in model.parameters()
)
print(total_params)

541


In [4]:
flops, params = clever_format([macs*2, params], "%.3f")

In [5]:
flops, params

('10.780K', '541.000B')

In [24]:
# norm_adj

In [16]:
from models.pedgraph_model_3 import *
norm_adj = get_norm_adj()
model = PedestrianGraph(device='cpu')

input = torch.randn(1, 16, 14, 2)
# output = model(data, norm_adj)
macs, params = profile(model, inputs=(input,norm_adj))

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register count_gru() for <class 'torch.nn.modules.rnn.GRU'>.


In [17]:
flops, params = clever_format([macs*2, params], "%.3f")

In [18]:
flops, params 

('748.064K', '12.497K')

In [21]:
from thop import profile
from thop.rnn_hooks import count_gru
import torch
import torch.nn as nn

# Patched GRU counter
def count_gru_fixed(m, x, y):
    # x is a tuple, but thop passes it weirdly with kwargs
    # Manually compute GRU flops
    batch_size = y[0].size(0)
    seq_len = y[0].size(1)
    input_size = m.input_size
    hidden_size = m.hidden_size
    num_layers = m.num_layers

    # Each GRU step: ~6 * hidden_size * (input_size + hidden_size) flops
    flops_per_step = 6 * hidden_size * (input_size + hidden_size)
    total_flops = flops_per_step * seq_len * batch_size * num_layers
    m.total_ops += torch.DoubleTensor([total_flops])

batch_size = 2

x1_dummy = [
    torch.randn(batch_size, 16, 1),
    torch.randn(batch_size, 16, 1),
]
x2_dummy = torch.randn(batch_size, 2)

model = SFGRU(x1_data_sizes=[(16, 1), (16, 1)], x2_data_size=(batch_size,2))
model.eval()

class SFGRUWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x1_0, x1_1, x2):
        return self.model([x1_0, x1_1], x2)

wrapper = SFGRUWrapper(model)

macs, params_raw = profile(
    wrapper,
    inputs=(x1_dummy[0], x1_dummy[1], x2_dummy),
    custom_ops={nn.GRU: count_gru_fixed}
)

flops, params = clever_format([macs / batch_size * 2, params_raw], "%.3f")
print(flops, params)

[INFO] Customize rule count_gru_fixed() <class 'torch.nn.modules.rnn.GRU'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
10.764K 223.000B


In [22]:
batch_size = 2

x1_dummy = [
    torch.randn(batch_size, 32, 1),
    torch.randn(batch_size, 32, 1),
]
x2_dummy = torch.randn(batch_size, 2)

model = SFGRU(x1_data_sizes=[(32, 1), (32, 1)], x2_data_size=(batch_size,2))
model.eval()

class SFGRUWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x1_0, x1_1, x2):
        return self.model([x1_0, x1_1], x2)

wrapper = SFGRUWrapper(model)

macs, params_raw = profile(
    wrapper,
    inputs=(x1_dummy[0], x1_dummy[1], x2_dummy),
    custom_ops={nn.GRU: count_gru_fixed}
)

flops, params = clever_format([macs / batch_size * 2, params_raw], "%.3f")
print(flops, params)

[INFO] Customize rule count_gru_fixed() <class 'torch.nn.modules.rnn.GRU'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
21.516K 223.000B
